# Nemotron-3-Nano-30B

In [1]:
!pip install --no-index \
    --find-links /kaggle/input/notebooks/deep262003/nvidia-nemotron-packages/wheels/ \
    unsloth trl peft accelerate bitsandbytes datasets \
    --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


In [2]:
MODEL_PATH = '/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1'

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1/",
    max_seq_length=8192,
    load_in_4bit=False,
    offload_folder="/tmp/offload",
    device_map="cuda",
)
print("Model ready")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.4: Fast Nemotron_H patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of `(selective_state_update, causal_conv1d_fn, causal_conv1d_update)` is None. Falling back to the naive implementation. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d


Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/401 [00:00<?, ?it/s]

/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1/ does not have a padding token! Will use pad_token = <SPECIAL_999>.
Model ready


# LoRA Adapter

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=False,
)
print("LoRA Ready")

LoRA Ready


# Data Loader

In [4]:
import pandas as pd

train_df = pd.read_csv("/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/test.csv")

print(train_df.shape)
train_df.head(3)

(9500, 3)


,id,prompt,answer
0,00066667,"In Alice's Wonderland, a secret bit manipulati...",10010111
1,000b53cf,"In Alice's Wonderland, a secret bit manipulati...",01000011
2,00189f6a,"In Alice's Wonderland, secret encryption rules...",cat imagines book


In [5]:
def format_fn(row):
    return f"""<|im_start|>user
    {row['prompt']}
    Think step by step. Put your final answer in \\boxed{{}}.<|im_end|>
    <|im_start|>assistant
    \\boxed{{{row['answer']}}}<|im_end|>"""

formatted = train_df.apply(format_fn, axis=1).tolist()
formatted[0]

"<|im_start|>user\n    In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n01010001 -> 11011101\n00001001 -> 01101101\n00010101 -> 01010101\n11111111 -> 10000001\n10011101 -> 01000101\n00111011 -> 00001001\n10111101 -> 00000101\n00100110 -> 10110011\n\nNow, determine the output for: 00110100\n    Think step by step. Put your final answer in \\boxed{}.<|im_end|>\n    <|im_start|>assistant\n    \\boxed{10010111}<|im_end|>"

In [6]:
from datasets import Dataset

dataset = Dataset.from_dict({'text': formatted})
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 8550
    })
    test: Dataset({
        features: ['text'],
        num_rows: 950
    })
})


# Training

In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text",
    max_seq_length=8192,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        bf16=True,
        output_dir="/kaggle/working/nemotron-lora",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_steps=100,
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/8550 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/950 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,550 | Num Epochs = 3 | Total steps = 3,207
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 3,735,552 of 31,581,672,896 (0.01% trained)


Step,Training Loss,Validation Loss
100,1.055640,0.898165
200,0.971858,0.862448
300,0.726050,0.847139
400,0.975379,0.836926
500,1.069727,0.829125
600,0.697869,0.824192
700,0.801052,0.815746
800,0.656080,0.811895
900,0.781902,0.805811
1000,0.866682,0.800401


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/nemotron-lora/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/nemotron-lora/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/nemotron-lora/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/nemotron-lora/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/nemotron-lora/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/nemotron-lora/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/nemotron-lora/checkpoint-700/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/nemotron-lora/checkpoint-800/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata 

TrainOutput(global_step=3207, training_loss=0.7844058170027178, metrics={'train_runtime': 15881.4954, 'train_samples_per_second': 1.615, 'train_steps_per_second': 0.202, 'total_flos': 9.022110890115878e+17, 'train_loss': 0.7844058170027178, 'epoch': 3.0})

In [8]:
model.save_pretrained("/kaggle/working/nemotron-lora")
tokenizer.save_pretrained("/kaggle/working/nemotron-lora")

# Zip
import shutil
shutil.make_archive("/kaggle/working/submission", "zip", "/kaggle/working/nemotron-lora")

print("Done — submission.zip ready")

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/nemotron-lora/tokenizer_config.json.


Done — submission.zip ready
